# 3. TRAINING & EMBEDDING DENGAN INDOBERT

**Jurnal: Ekstraksi Kata Kunci dengan N-Gram dan IndoBERT untuk Rekomendasi Wisata Bogor**

---

## Alur:
```
Tokenizer IndoBERT
       ↓
Training IndoBERT with SimCSE + MultipleNegativesRankingLoss (uses Cosine Similarity)
       ↓
Generate Embedding Description
       ↓
Save Embedding (.npy format)
```

In [1]:
!pip install torch transformers tqdm -q


[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm
from IPython.display import display
import os
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = './data/'
MODEL_PATH = './models/'
os.makedirs(MODEL_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

env_info = pd.DataFrame({
    'Parameter': ['Device', 'CUDA Available'],
    'Nilai': [str(device), torch.cuda.is_available()]
})
print("TABEL: ENVIRONMENT")
display(env_info)

c:\laragon\bin\python\python-3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TABEL: ENVIRONMENT


,Parameter,Nilai
0,Device,cpu
1,CUDA Available,False


In [3]:
# Load data
df = pd.read_csv(f'{DATA_PATH}data_with_keywords.csv')

# Handle NaN - PENTING!
df['deskripsi_clean'] = df['deskripsi_clean'].fillna('').astype(str)
# Filter empty strings
df = df[df['deskripsi_clean'].str.strip() != ''].reset_index(drop=True)

data_info = pd.DataFrame({
    'Keterangan': ['Total Data Valid', 'Jumlah Kategori'],
    'Nilai': [len(df), df['kategori'].nunique()]
})
print("TABEL: DATA DIMUAT")
display(data_info)

TABEL: DATA DIMUAT


,Keterangan,Nilai
0,Total Data Valid,295
1,Jumlah Kategori,7


## 3.1 Tokenizer IndoBERT

In [4]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

print(f"🔄 Loading IndoBERT: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_info = pd.DataFrame({
    'Parameter': ['Model Name', 'Vocab Size', 'Max Length'],
    'Nilai': [MODEL_NAME, tokenizer.vocab_size, tokenizer.model_max_length]
})
print("\nTABEL: TOKENIZER INFO")
display(tokenizer_info)

🔄 Loading IndoBERT: indobenchmark/indobert-base-p1...

TABEL: TOKENIZER INFO


,Parameter,Nilai
0,Model Name,indobenchmark/indobert-base-p1
1,Vocab Size,30521
2,Max Length,1000000000000000019884624838656


## 3.2 Setup Training Data (Contrastive Learning)

In [5]:
def create_training_pairs(df):
    """Create positive pairs from items with same category"""
    pairs = []
    categories = df['kategori'].unique()
    
    for cat in categories:
        cat_items = df[df['kategori'] == cat]['deskripsi_clean'].tolist()
        # Filter valid strings
        cat_items = [str(item).strip() for item in cat_items if pd.notna(item) and str(item).strip()]
        
        for i in range(len(cat_items)):
            for j in range(i+1, len(cat_items)):
                if cat_items[i] and cat_items[j]:  # Double check not empty
                    pairs.append((cat_items[i], cat_items[j]))
    
    return pairs

training_pairs = create_training_pairs(df)

pairs_info = pd.DataFrame({
    'Keterangan': ['Total Training Pairs', 'Jumlah Kategori'],
    'Nilai': [len(training_pairs), df['kategori'].nunique()]
})
print("TABEL: TRAINING PAIRS")
display(pairs_info)

# Show sample
if training_pairs:
    print(f"\nContoh pair: '{training_pairs[0][0][:50]}...' ↔ '{training_pairs[0][1][:50]}...'")

TABEL: TRAINING PAIRS


,Keterangan,Nilai
0,Total Training Pairs,9381
1,Jumlah Kategori,7



Contoh pair: 'curug ciampea bogor adalah salah satu pesona alam ...' ↔ 'bukit cirimpak salah satu camping ground yang ada ...'


## 3.3 Training IndoBERT dengan SimCSE + MultipleNegativesRankingLoss

In [6]:
# Load model
print("🔄 Loading IndoBERT Model...")
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)

model_info = pd.DataFrame({
    'Parameter': ['Model Name', 'Hidden Size', 'Num Layers'],
    'Nilai': [MODEL_NAME, model.config.hidden_size, model.config.num_hidden_layers]
})
print("\nTABEL: MODEL INFO")
display(model_info)

🔄 Loading IndoBERT Model...

TABEL: MODEL INFO


,Parameter,Nilai
0,Model Name,indobenchmark/indobert-base-p1
1,Hidden Size,768
2,Num Layers,12


In [7]:
# Hyperparameters
BATCH_SIZE = 4  # Smaller batch for stability
EPOCHS = 3
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
TEMPERATURE = 0.05

hyperparams = pd.DataFrame({
    'Hyperparameter': ['Batch Size', 'Epochs', 'Learning Rate', 'Max Length', 'Temperature'],
    'Nilai': [BATCH_SIZE, EPOCHS, LEARNING_RATE, MAX_LENGTH, TEMPERATURE]
})
print("TABEL: HYPERPARAMETERS")
display(hyperparams)

TABEL: HYPERPARAMETERS


,Hyperparameter,Nilai
0,Batch Size,4.00000
1,Epochs,3.00000
2,Learning Rate,0.00002
3,Max Length,128.00000
4,Temperature,0.05000


In [8]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def multiple_negatives_ranking_loss(anchor_emb, positive_emb, temperature=0.05):
    """
    MultipleNegativesRankingLoss untuk SimCSE.
    Uses Cosine Similarity.
    """
    anchor_emb = F.normalize(anchor_emb, p=2, dim=1)
    positive_emb = F.normalize(positive_emb, p=2, dim=1)
    
    # Cosine similarity
    similarity_matrix = torch.matmul(anchor_emb, positive_emb.T) / temperature
    labels = torch.arange(similarity_matrix.size(0)).to(device)
    loss = F.cross_entropy(similarity_matrix, labels)
    
    return loss

print("✅ Loss function ready: MultipleNegativesRankingLoss (uses Cosine Similarity)")

✅ Loss function ready: MultipleNegativesRankingLoss (uses Cosine Similarity)


In [ ]:
# Simple training loop without DataLoader
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("\n" + "="*60)
print("🚀 TRAINING INDOBERT dengan SimCSE + MultipleNegativesRankingLoss")
print("="*60)

training_log = []
model.train()

# Limit pairs for faster training
max_pairs = min(len(training_pairs), 500)
pairs_to_use = training_pairs[:max_pairs]

for epoch in range(EPOCHS):
    total_loss = 0
    num_batches = 0
    
    # Batch manually
    for i in tqdm(range(0, len(pairs_to_use), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}"):
        batch = pairs_to_use[i:i+BATCH_SIZE]
        if len(batch) < 2:  # Need at least 2 for contrastive
            continue
        
        anchors = [str(p[0]) for p in batch]
        positives = [str(p[1]) for p in batch]
        
        # Tokenize
        anchor_enc = tokenizer(anchors, padding=True, truncation=True, 
                               max_length=MAX_LENGTH, return_tensors='pt').to(device)
        positive_enc = tokenizer(positives, padding=True, truncation=True,
                                 max_length=MAX_LENGTH, return_tensors='pt').to(device)
        
        # Forward
        anchor_output = model(**anchor_enc)
        positive_output = model(**positive_enc)
        
        anchor_emb = mean_pooling(anchor_output, anchor_enc['attention_mask'])
        positive_emb = mean_pooling(positive_output, positive_enc['attention_mask'])
        
        # Loss
        loss = multiple_negatives_ranking_loss(anchor_emb, positive_emb, TEMPERATURE)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    avg_loss = total_loss / max(num_batches, 1)
    training_log.append({'Epoch': epoch+1, 'Avg Loss': round(avg_loss, 4)})
    print(f"   Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

print("\n✅ Training completed!")


🚀 TRAINING INDOBERT dengan SimCSE + MultipleNegativesRankingLoss


Epoch 1/3: 100%|██████████| 125/125 [09:35<00:00,  4.60s/it]


   Epoch 1 - Average Loss: 1.3940


Epoch 2/3: 100%|██████████| 125/125 [09:26<00:00,  4.53s/it]


   Epoch 2 - Average Loss: 1.3864


Epoch 3/3:  66%|██████▋   | 83/125 [07:45<03:58,  5.69s/it]

In [ ]:
# Training log
log_df = pd.DataFrame(training_log)
print("TABEL: TRAINING LOG")
display(log_df)

TABEL: TRAINING LOG


,Epoch,Avg Loss
0,1,1.3942
1,2,1.3863
2,3,1.3863


In [ ]:
# Save model
model.save_pretrained(f'{MODEL_PATH}indobert-simcse-trained')
tokenizer.save_pretrained(f'{MODEL_PATH}indobert-simcse-trained')

print(f"✅ Model saved to: {MODEL_PATH}indobert-simcse-trained")

✅ Model saved to: ./models/indobert-simcse-trained


## 3.4 Generate Embedding dengan Model Trained

In [ ]:
model.eval()

def get_embedding(text):
    if not text or not str(text).strip():
        return np.zeros(768)
    
    encoded = tokenizer(
        str(text), padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        output = model(**encoded)
        embedding = mean_pooling(output, encoded['attention_mask'])
    
    return embedding.cpu().numpy().squeeze()

print("✅ Embedding function ready")

✅ Embedding function ready


In [ ]:
# Generate embeddings
print("🔄 Generating embeddings...\n")

# Reload original data
df_full = pd.read_csv(f'{DATA_PATH}data_with_keywords.csv')
df_full['deskripsi_clean'] = df_full['deskripsi_clean'].fillna('').astype(str)

embeddings = []
for text in tqdm(df_full['deskripsi_clean'], desc="Generating"):
    emb = get_embedding(text)
    embeddings.append(emb)

embeddings = np.array(embeddings)

emb_info = pd.DataFrame({
    'Parameter': ['Total Embeddings', 'Shape', 'Dimension'],
    'Nilai': [len(embeddings), str(embeddings.shape), embeddings.shape[1]]
})
print("\nTABEL: EMBEDDING INFO")
display(emb_info)

🔄 Generating embeddings...




Generating:   0%| | 0


Generating:   0%| | 1


Generating:   1%| | 2


Generating:   1%| | 3

...


Generating: 100%|▉| 2


Generating: 100%|█| 2


Generating: 100%|█| 2


TABEL: EMBEDDING INFO


,Parameter,Nilai
0,Total Embeddings,296
1,Shape,"(296, 768)"
2,Dimension,768


## 3.5 Save Embedding (.npy format)

In [ ]:
np.save(f'{DATA_PATH}indobert_embeddings.npy', embeddings)

saved = pd.DataFrame({
    'File': ['indobert_embeddings.npy'],
    'Shape': [str(embeddings.shape)],
    'Size': [f"{embeddings.nbytes/1024/1024:.2f} MB"]
})
print("TABEL: EMBEDDING SAVED")
display(saved)

TABEL: EMBEDDING SAVED


,File,Shape,Size
0,indobert_embeddings.npy,"(296, 768)",1.73 MB


In [ ]:
# Summary
summary = pd.DataFrame({
    'Aspek': ['Model', 'Training Method', 'Loss Function', 'Similarity', 'Epochs', 'Total Embeddings'],
    'Nilai': [MODEL_NAME, 'SimCSE', 'MultipleNegativesRankingLoss', 'Cosine Similarity',
              EPOCHS, len(embeddings)]
})
print("\nTABEL: SUMMARY")
display(summary)


TABEL: SUMMARY


,Aspek,Nilai
0,Model,indobenchmark/indobert-base-p1
1,Training Method,SimCSE
2,Loss Function,MultipleNegativesRankingLoss
3,Similarity,Cosine Similarity
4,Epochs,3
5,Total Embeddings,296
